# NexusAI — Unsloth Fine-Tuning (Step 1 → Step 12)

Yeh notebook Llama-3 (8B, 4-bit) ko **Unsloth + LoRA + TRL SFTTrainer** se fine-tune karta hai.

**Important rules (sab errors yahi se aate hain):**
1. `import unsloth` SABSE PEHLE karna hai — `trl` / `transformers` / `peft` ke pehle.
2. Naye `trl` me `tokenizer=` deprecated hai → `processing_class=` use karo.
3. `dataset_text_field`, `max_seq_length`, `packing` ab `SFTConfig` me jaate hain (na ki `TrainingArguments` me).
4. Step 1 ke baad **Runtime → Restart session** zaroori hai.

## Step 1 — Install dependencies

Compatible versions install karte hain. Iske baad **runtime restart karna ZAROORI hai** (Colab: Runtime → Restart session).

In [ ]:
%%capture
# Latest Unsloth + compatible transformers/trl/peft stack
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-cache-dir "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
!pip install --upgrade --no-cache-dir "transformers>=4.46.0" "trl>=0.12.0" "peft>=0.13.0" "accelerate>=1.0.0" "bitsandbytes>=0.44.0" "datasets>=2.20.0"

print("\nInstallation done. RESTART RUNTIME ab. (Runtime -> Restart session)")

## Step 2 — Load base model + tokenizer (4-bit)

`import unsloth` SABSE PEHLE — yeh patches lagata hai jo trl/transformers/peft ko optimize karte hain.

In [ ]:
import unsloth  # <-- MUST be first
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

max_seq_length = 2048   # Unsloth RoPE scaling support karta hai, koi bhi value chal jayegi
dtype = None            # None = auto-detect. Float16 for T4/V100, Bfloat16 for Ampere+
load_in_4bit = True     # 4bit quantization for memory saving

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length  = max_seq_length,
    dtype           = dtype,
    load_in_4bit    = load_in_4bit,
)

print("Model loaded:", model.config._name_or_path)

## Step 3 — Attach LoRA adapters

Sirf chhote LoRA adapters train karenge, base model frozen rahega → fast + memory-efficient.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,                                # LoRA rank: 8/16/32/64/128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0,                                 # 0 = optimized
    bias           = "none",                            # "none" = optimized
    use_gradient_checkpointing = "unsloth",            # very long context support
    random_state   = 3407,
    use_rslora     = False,
    loftq_config   = None,
)

model.print_trainable_parameters()

## Step 4 — Load dataset

**Teen options hain — apni need ke hisaab se ek choose karo:**

- **Step 4A** — Apna `.jsonl` Colab me upload karke train karo  ⭐ (most flexible)
- **Step 4B** — Drive ya local path se `.jsonl` load karo
- **Step 4C** — Public HuggingFace dataset use karo (e.g. `yahma/alpaca-cleaned`)

Sab options ka final output ek hi hota hai: `dataset` jisme `text` column hota hai. Tumhara `.jsonl` jis bhi format me ho (Alpaca / Chat / Plain), ye auto-detect karta hai.

**Supported `.jsonl` formats:**
```jsonl
// Format 1 — Alpaca
{"instruction": "...", "input": "...", "output": "..."}

// Format 2 — Prompt/Completion
{"prompt": "...", "completion": "..."}

// Format 3 — Chat / Messages
{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}

// Format 4 — Plain text (already formatted)
{"text": "..."}
```

### Common helpers — alpaca prompt template + auto-formatter

Ye cell SAB cases ke liye chalana hai (4A / 4B / 4C). Iske baad neeche se koi ek sub-step chuno.

In [ ]:
from datasets import load_dataset, Dataset

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token  # warna model generation rokega nahi

def auto_format_dataset(ds):
    """Apne dataset ka format auto-detect karke `text` column generate karta hai."""
    cols = set(ds.column_names)

    # Case 1: Already formatted
    if "text" in cols:
        print("[auto-format] 'text' column mil gaya, as-is use kar rahe hain.")
        return ds

    # Case 2: Alpaca format
    if {"instruction", "output"}.issubset(cols):
        print("[auto-format] Alpaca format detect hua (instruction/input/output).")
        def fmt(ex):
            instr  = ex.get("instruction", "") or ""
            inp    = ex.get("input", "") or ""
            out    = ex.get("output", "") or ""
            return {"text": alpaca_prompt.format(instr, inp, out) + EOS_TOKEN}
        return ds.map(fmt, remove_columns=ds.column_names)

    # Case 3: prompt/completion
    if {"prompt", "completion"}.issubset(cols):
        print("[auto-format] prompt/completion format detect hua.")
        def fmt(ex):
            return {"text": alpaca_prompt.format(ex["prompt"], "", ex["completion"]) + EOS_TOKEN}
        return ds.map(fmt, remove_columns=ds.column_names)

    # Case 4: messages (ChatML / OpenAI style)
    if "messages" in cols:
        print("[auto-format] messages format detect hua, chat template apply kar rahe hain.")
        def fmt(ex):
            text = tokenizer.apply_chat_template(
                ex["messages"], tokenize=False, add_generation_prompt=False
            )
            return {"text": text}
        return ds.map(fmt, remove_columns=ds.column_names)

    # Case 5: question/answer
    if {"question", "answer"}.issubset(cols):
        print("[auto-format] question/answer format detect hua.")
        def fmt(ex):
            return {"text": alpaca_prompt.format(ex["question"], "", ex["answer"]) + EOS_TOKEN}
        return ds.map(fmt, remove_columns=ds.column_names)

    raise ValueError(
        f"Dataset format samajh nahi aaya. Columns: {ds.column_names}. "
        "Manually 'text' column banao ya supported format me convert karo."
    )

print("Helpers ready. Ab Step 4A / 4B / 4C me se ek chalao.")

### Step 4A — ⭐ Apna `.jsonl` Colab me upload karo

Cell run karoge toh **"Choose Files"** button aayega — apni `.jsonl` file select karo. Ek se zyada files bhi select kar sakte ho (sab merge ho jayengi).

In [ ]:
from google.colab import files
import os

uploaded = files.upload()   # browser me file picker khulega

jsonl_paths = [name for name in uploaded.keys() if name.endswith(".jsonl") or name.endswith(".json")]
if not jsonl_paths:
    raise ValueError("Koi .jsonl/.json file upload nahi hui. Dobara try karo.")

print("Uploaded files:", jsonl_paths)
for p in jsonl_paths:
    size_mb = os.path.getsize(p) / 1024 / 1024
    print(f"  {p}  ({size_mb:.2f} MB)")

raw_dataset = load_dataset("json", data_files=jsonl_paths, split="train")
print("\nRaw rows :", len(raw_dataset))
print("Columns  :", raw_dataset.column_names)
print("Sample   :", raw_dataset[0])

dataset = auto_format_dataset(raw_dataset)
print("\nFinal dataset size:", len(dataset))
print("Sample text:\n", dataset[0]["text"][:500], "...")

### Step 4B — Local path / Google Drive se `.jsonl` load karo (alternative)

Agar file pehle se Colab me hai (e.g. Drive mount kiya hua, ya `/content/data.jsonl`), to ye use karo. Agar 4A chala liya hai to ise SKIP karo.

In [ ]:
# Drive mount karna ho to uncomment:
# from google.colab import drive
# drive.mount("/content/drive")

JSONL_PATH = "/content/data.jsonl"   # apni file ka path daalo

raw_dataset = load_dataset("json", data_files=JSONL_PATH, split="train")
print("Raw rows :", len(raw_dataset))
print("Columns  :", raw_dataset.column_names)

dataset = auto_format_dataset(raw_dataset)
print("\nFinal dataset size:", len(dataset))
print("Sample text:\n", dataset[0]["text"][:500], "...")

### Step 4C — Public HuggingFace dataset (alternative)

Apna dataset nahi hai? Public Alpaca dataset use kar lo. Agar 4A ya 4B chala liya hai to ise SKIP karo.

In [ ]:
raw_dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset = auto_format_dataset(raw_dataset)

print("Final dataset size:", len(dataset))
print("Sample text:\n", dataset[0]["text"][:500], "...")

## Step 5 — Quick sanity check on a sample

Tokenizer aur prompt format theek se kaam kar rahe hain ya nahi — verify.

In [ ]:
sample = dataset[0]["text"]
tokens = tokenizer(sample, return_tensors="pt")

print("Token count :", tokens.input_ids.shape[1])
print("First 30 ids:", tokens.input_ids[0][:30].tolist())
print("EOS present :", tokenizer.eos_token_id in tokens.input_ids[0].tolist())

## Step 6 — Configure SFTTrainer

**Yahi par tumhari pichli error aayi thi.** Naya pattern:
- `tokenizer=` ❌  →  `processing_class=` ✅
- `TrainingArguments` ❌  →  `SFTConfig` ✅ (training args + sft args dono ek jagah)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,            # naya naam (tokenizer= deprecated)
    train_dataset    = dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = max_seq_length,
        dataset_num_proc            = 2,
        packing                     = False,    # True kar do short sequences ke liye 5x faster
        padding_free                = False,    # naya trl me default True hai, max_length ke saath conflict karta hai
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps                = 5,
        max_steps                   = 60,        # demo ke liye 60. Full epoch chahiye toh num_train_epochs use karo
        # num_train_epochs          = 1,
        learning_rate               = 2e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 1,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 3407,
        output_dir                  = "outputs",
        report_to                   = "none",   # wandb/tensorboard chahiye to yahan badlo
    ),
)

print("Trainer ready.")

## Step 7 — GPU memory snapshot (before training)

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory       = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU             = {gpu_stats.name}")
print(f"Max memory      = {max_memory} GB")
print(f"Reserved before = {start_gpu_memory} GB")

## Step 8 — Train the model

Yahi step pe pehle `TypeError: tokenizer` aur `ImportError: CompileConfig` aaye the. Step 1 + Step 6 ke fixes ke baad ye ab clean chalega.

In [ ]:
trainer_stats = trainer.train()

## Step 9 — Training stats (time + memory used)

In [ ]:
used_memory          = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage      = round(used_memory          / max_memory * 100, 3)
lora_percentage      = round(used_memory_for_lora / max_memory * 100, 3)

print(f"Training time           = {trainer_stats.metrics['train_runtime']} sec")
print(f"                        = {round(trainer_stats.metrics['train_runtime']/60, 2)} min")
print(f"Peak reserved memory    = {used_memory} GB")
print(f"  ...used by training   = {used_memory_for_lora} GB")
print(f"Peak memory  % of max   = {used_percentage} %")
print(f"Training mem % of max   = {lora_percentage} %")

## Step 10 — Inference (test trained model)

`FastLanguageModel.for_inference(model)` 2x faster generation ke liye optimize karta hai.

In [ ]:
FastLanguageModel.for_inference(model)

instruction = "Continue the fibonnaci sequence."
input_text  = "1, 1, 2, 3, 5, 8"

inputs = tokenizer(
    [alpaca_prompt.format(instruction, input_text, "")],
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 128,
    use_cache      = True,
)

print(tokenizer.batch_decode(outputs)[0])

### Step 10b — Streaming output (optional, dekhne me cool lagta hai)

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

inputs = tokenizer(
    [alpaca_prompt.format("What is a famous tall tower in Paris?", "", "")],
    return_tensors="pt",
).to("cuda")

streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer=streamer, max_new_tokens=128)

## Step 11 — Save the fine-tuned model

3 options — apni need ke hisaab se uncomment karo:

1. **LoRA adapters only** (chhota, ~100 MB) — base model alag se chahiye load karne ko.
2. **Merged 16-bit** — full standalone model.
3. **Push to Hugging Face Hub** — token chahiye.

In [ ]:
# Option 1: LoRA adapters only (recommended for fast iteration)
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("LoRA adapters saved -> ./lora_model")

# Option 2: Merged 16-bit (uncomment to use)
# model.save_pretrained_merged("nexusai_merged_16bit", tokenizer, save_method="merged_16bit")

# Option 3: Push to HF Hub (uncomment + add token)
# model.push_to_hub("your-username/nexusai-lora", token="hf_xxx")
# tokenizer.push_to_hub("your-username/nexusai-lora", token="hf_xxx")

## Step 12 — Reload saved model and verify

Saved adapters wapas load karke test karte hain ki sab kuch sahi save hua.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "lora_model",     # Step 11 wala folder
    max_seq_length = max_seq_length,
    dtype          = dtype,
    load_in_4bit   = load_in_4bit,
)
FastLanguageModel.for_inference(model)

inputs = tokenizer(
    [alpaca_prompt.format(
        "What is the capital of India?",
        "",
        "",
    )],
    return_tensors="pt",
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

print("\nSab steps complete. NexusAI fine-tuned model ready hai.")